# 第 12 章实验：合成圆形与极小 U-Net
在内存中生成图像和二值 mask，观察 `[N,1,H,W]` 契约、skip 拼接与像素级输出。

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(12)
size = 32
samples = 32
train_steps = 60

In [ ]:
yy, xx = torch.meshgrid(torch.arange(size), torch.arange(size), indexing='ij')
images, masks = [], []
for i in range(samples):
    cx = int(torch.randint(9, 23, (1,)))
    cy = int(torch.randint(9, 23, (1,)))
    radius = int(torch.randint(4, 8, (1,)))
    mask = (((xx-cx)**2 + (yy-cy)**2) <= radius**2).float()
    image = (0.15 + 0.75 * mask + 0.10 * torch.randn(size, size)).clamp(0, 1)
    images.append(image); masks.append(mask)
images = torch.stack(images)[:, None]
masks = torch.stack(masks)[:, None]
print('images:', tuple(images.shape), images.dtype, 'masks:', tuple(masks.shape), torch.unique(masks).tolist())

In [ ]:
class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.Conv2d(4, 4, 3, padding=1), nn.ReLU())
        self.bottleneck = nn.Sequential(nn.Conv2d(4, 8, 3, padding=1), nn.ReLU())
        self.dec = nn.Sequential(nn.Conv2d(12, 4, 3, padding=1), nn.ReLU(), nn.Conv2d(4, 1, 1))
    def forward(self, x):
        skip = self.enc(x)
        low = self.bottleneck(F.max_pool2d(skip, 2))
        up = F.interpolate(low, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.dec(torch.cat([up, skip], dim=1))

model = TinyUNet()
with torch.no_grad():
    probe = model(images[:2])
print('input:', tuple(images[:2].shape), 'logits:', tuple(probe.shape))

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
history = []
for _ in range(train_steps):
    optimizer.zero_grad()
    logits = model(images)
    loss = F.binary_cross_entropy_with_logits(logits, masks)
    loss.backward(); optimizer.step(); history.append(float(loss.detach()))
with torch.no_grad():
    probs = model(images).sigmoid()
    prediction = (probs > 0.5).float()
    intersection = (prediction * masks).sum()
    dice = float((2 * intersection + 1e-6) / (prediction.sum() + masks.sum() + 1e-6))
print(f'final BCE={history[-1]:.4f}, hard Dice={dice:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(10, 2.6))
axes[0].imshow(images[0,0], cmap='gray'); axes[0].set_title('image')
axes[1].imshow(masks[0,0], cmap='gray'); axes[1].set_title('target')
axes[2].imshow(probs[0,0], cmap='viridis', vmin=0, vmax=1); axes[2].set_title('probability')
axes[3].plot(history); axes[3].set_title('BCE loss')
for ax in axes[:3]: ax.axis('off')
plt.tight_layout(); plt.show()

## 破坏实验建议
移除 `skip`、把 mask 用双线性插值缩放，或把圆半径减到 1–2 像素。记录边界、Dice 和全背景预测的变化；不要只看 pixel accuracy。